In [ ]:
"""LIBRERIAS Y DEPENDENCIAS"""
import pandas as pd
import numpy as np #trabaja con arrays
import matplotlib.pyplot as plt #parte grafica
import seaborn as sns #librería de visualización construida sobre Matplotlib y sirve para buscar correlaciones entre variables
import csv
import sqlite3


from sklearn.model_selection import train_test_split #divide el dataset en train y test
"""NOTA IMPORTANTE, TENGO QUE DIVIDIR BIEN EL DATASET."""
from sklearn.preprocessing import StandardScaler #estandariza las variables numéricas.
"""NOTA IMPORTANTE fit solamente sobre X_train."""
from sklearn.ensemble import RandomForestClassifier #modelo basado en muchos árboles de decisión.
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
#modelo avanzado de redes neuronales, que permite la clasificación de datos no lineales.

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_curve, auc, confusion_matrix
#curva ROC: Permite analizar cómo cambia el comportamiento del modelo cuando modificás el umbral de decisión.
#auc calcula el área bajo la curva ROC, que es una métrica de desempeño del modelo.
"""confusion_matrix util y usado en ciberseguridad
permite ver la cantidad de falsos positivos y falsos negativos que tiene el modelo, 
lo cual es muy importante en ciberseguridad, 
ya que un falso positivo puede generar alertas innecesarias y un falso negativo puede 
permitir que un ataque pase desapercibido."""

from sklearn.preprocessing import label_binarize
#Convierte etiquetas categóricas/multiclase en representación binaria.
from itertools import cycle

"""FLUJO INICIAL PROBLABLE
Random Forest + Logistic Regression + SVM + MLP"""


In [ ]:
"""CONEXIONES A LA BASE DE DATOS"""
df_ddos_2 = pd.read_csv ("/home/rhev/Analisis de datos/Trabajo-Final-Analisis-de-Datos-1/DDoSdata.csv")
print(df_ddos_2.head(3).T)
df_ddos_2["stime"] = pd.to_datetime(df_ddos_2["stime"], unit="s")

In [ ]:
"""RENOMBRANDO COLUMNAS"""
df_ddos_2 = df_ddos_2.rename(columns={
    'pkSeqID': 'id',
    'stime': 'start_time',
    'flgs': 'flags',
    'flgs_number': 'flags_number',
    'proto': 'protocol',
    'proto_number': 'protocol_number',
    'saddr': 'source_address',
    'sport': 'source_port',
    'daddr': 'destination_address',
    'dport': 'destination_port',
    'pkts': 'flow_packets',
    'Itime': 'initial_time',
    'dur': 'duration',
    'Itime' : 'initial_time',
    #####################################################################
    'spkts' : 'source_packets',
    'dpkts' : 'destination_packets',
    #relacion entre source_packets y destination_packets es importante
    'sbytes' : 'source_bytes',
    'dbytes' : 'destination_bytes',
    #esta relacion es como la anterior, la alta frecuencia incial seguida por una baja secuencia
    #de respuesta indica un flujo anomalo de paquetes
    #'Rate' : cantidad de trafico / tiempo
    'srate' : 'source_rate',
    #la presencia de muchas fuentes con tasas elevadas puede ser una señal muy interesante.
    'drate' : 'destination_rate',
    #La diferencia entre ambas puede ser significativa.
    'N_IN_Conn_P_DstIP' : 'num_incoming.connection_per_destip',
    #número de conexiones entrantes asociadas a una IP de destino.
    #los ataques ddos se caracterizan por atacar una misma victina desde multiples IP
    'N_IN_Conn_P_SrcIP' : 'num_incoming.connection_per_sourceip',
    #una misma IP participando en muchas conexiones puede ser relevante.
    'TnP_PSrcIP' : 'total_packets_from_sourceip',
    #representa el total de paquetes asociado a una IP de origen y cuanto trafico genera una ip
    'TnP_PDstIP' : 'total_packets_from_destip',
    #lo mismo pero al inverso
    'TnP_PerProto' : 'total_packets_per_protocol',
    #cantidad total de paquetes asociados a ese protocolo. TCP, UDP, ICMP, etc.
    'TnP_Per_Dport' : 'total_packets_per_destination_port',
    #cantidad total de paquetes asociados a ese puerto de destino.
    'AR_P_Proto_P_Sport' : 'average_rate_per_protocol_per_sourceport',
    'AR_P_Proto_P_Dport' : 'average_rate_per_protocol_per_destinationport',
    'Pkts_P_State_P_Protocol_P_DestIP' : 'pks_per_state_per_protocol_per_destip',
    'Pkts_P_State_P_Protocol_P_SrcIP' : 'pks_per_state_per_protocol_per_sourceip',
})   
df_ddos_2.columns

In [ ]:
"""ANALISIS INICIAL DE DATOS"""

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


print("Información del dataset:")
# display(df_ddos_2.shape)
# display(df_ddos_2.info())

# print("Descripción estadística del dataset:")
# display(df_ddos_2.describe())

print("Primeras 5 filas del dataset:")
display(df_ddos_2.head(100))

# print("Últimas 5 filas del dataset:")
# display(df_ddos_2.tail(5))

# print("Tipos de datos del dataset:")
# print(df_ddos_2.dtypes)

# print("Nombres de las columnas del dataset:")
# display(df_ddos_2.columns)


Observaciones Iniciales:
1) El dataset proviene de Kaggle, especificamente https://www.kaggle.com/datasets/siddharthm1698/ddos-botnet-attack-on-iot-devices.
2) Cuenta con 1.927.101 registros distribuidos en 




1) 192.168.100.150, en los primeros 20 registros ataca desde 20 puertos distintos, en los 100 registros
   posteriores se observan 3 IP's distintas pertenecientes a la misma subred.
2) La IP objetivo es la misma en los 20 primeros casos, 192.168.100.3.
3) El puerto objetivo es el n°80, puerto estandar de las conexiones HTTP sin cifrar.
4) Las primeras 20 son emitidas y recibidas al mismo tiempo.
4) El promedio de bytes emitidos y reenviados es de 1300 aproximados.
5) El agrupamiento por protocolo y el agrupamiento por puerto de destino están capturando cantidades diferentes de tráfico.





In [ ]:
"""Limpieza dataset ddos_2 basado en el video de
https://www.youtube.com/watch?v=QqEdSk98Wqs KNOWLEDGE DOCTOR"""
print("Dimensiones:", df_ddos_2.shape) #Dimensiones: (1927101, 49)

print("\nTipos de datos:")
print(df_ddos_2.dtypes)

print("\nValores nulos:") #sin valores nulos.
print(df_ddos_2.isnull().sum())

print("\nDuplicados:") #sin duplicados.
print(df_ddos_2.duplicated().sum())

print("\nValores únicos:") #1927101
print(df_ddos_2.nunique().sort_values())

print(df_ddos_2["destination_port"].duplicated().sum())
#aunque no hay duplicados, si hay muchos valores repetidos en la columna destination_port, lo cual puede afectar el modelo.

print("\nColumnas:")
df_ddos_2.columns

print("Se mostrara graficamente la cantidad de valores nulos por columna del dataset ddos_2")
def Missingvalues(df_ddos_2):
    missing_values = df_ddos_2.isnull().sum()
    fig = plt.figure(figsize=(8, 1))
    missing_values.plot(kind='bar')
    plt.title("Missing Values per Column")
    plt.xlabel("Columns")
    plt.ylabel("Total Number of Missing Values")
    plt.show()

Missingvalues(df_ddos_2)

Observaciones de la limpieza:

1) No tenemos valores nulos, pero la repeticion de valores como por ejemplo los puerto (80) pueden alterar 
   la capacidad de reconocimiento de patrones del Machine Learning.
   Su eliminacion por otra parte, tampoco es conveniente al ser una etiqueta fundamental del ataque ddos.
2) Tampoco tenemos que preocuparnos por una identacion inicial entre columnas que alteren el flujo de datos.

In [ ]:
"""GRAFICO DE CORRELACIONES NORMAL"""
# Mostrar todas las filas y columnas de pandas
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# Ver las primeras 5 filas del tráfico normal
# df_ddos_2[df_ddos_2["category"] == "Normal"].head(5)

# FILTRAR ÚNICAMENTE EL TRÁFICO NORMAL
df_normal = df_ddos_2[
    df_ddos_2["category"] == "Normal"
].copy()

# VARIABLES A UTILIZAR EN LA CORRELACIÓN
variables_correlacion = [
    "flow_packets",
    "bytes",
    "source_packets",
    "destination_packets",
    "rate",
    "total_packets_from_sourceip",
    "total_packets_from_destip",
    "total_packets_per_protocol",
    "total_packets_per_destination_port",
    "num_incoming.connection_per_destip",
    "num_incoming.connection_per_sourceip",
]

# NOMBRES MÁS LEGIBLES PARA LA MATRIZ
nombres_legibles = {
    "flow_packets": "P.Flujo",
    "bytes": "Total.B",
    "source_packets": "P. de Origen",
    "destination_packets": "P. de Destino",

    "rate": "Tasa de Tráfico",

    "total_packets_from_sourceip": "P. Por Ip de Origen",
    "total_packets_from_destip": "P. Por Ip de Destino",

    "total_packets_per_protocol": "P. Por Protocolo",
    "total_packets_per_destination_port": "P. Por Puerto Destino",

    "num_incoming.connection_per_destip": "Conect. por IP de Destino",
    "num_incoming.connection_per_sourceip": "Conect. por IP de Origen",
}

# MATRIZ DE CORRELACIÓN DEL TRÁFICO NORMAL
matriz_correlacion_normal = df_normal[
    variables_correlacion
].corr()

# Cambiar los nombres de filas y columnas
matriz_correlacion_normal = matriz_correlacion_normal.rename(
    index=nombres_legibles,
    columns=nombres_legibles
)
display(matriz_correlacion_normal)



# GRAFICAR
plt.figure(figsize=(16, 12))
sns.heatmap(
    matriz_correlacion_normal,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title(
    "Matriz de correlación de variables asociadas al tráfico Normal"
)
plt.tight_layout()



# GUARDAR EN SVG
plt.savefig(
    "matriz_correlacion_normal.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()

In [ ]:
"""GRAFICO DE CORRELACIONES DDoS"""

matriz_correlacion = df_ddos_2[variables_correlacion].corr()
matriz_correlacion = matriz_correlacion.rename(
    index=nombres_legibles,
    columns=nombres_legibles
)

plt.figure(figsize=(16, 12))
sns.heatmap(
    matriz_correlacion,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

#Guardado de la imagen en formato SVG
plt.title("Matriz de correlación de variables asociadas al tráfico DDoS")
plt.tight_layout()
plt.savefig(
    "matriz_correlacion.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()

In [ ]:
"""GRAFICO DE PUERTOS DE DESTINO Y CANTIDAD DE PAQUETES ASOCIADOS A CADA UNO DEL DATASET ddos_2"""
df_puertos = df_ddos_2.groupby("destination_port").size().reset_index(name="count")
print(df_puertos.head(10))

print("Se mostrara graficamente la cantidad de puertos de destino y la cantidad de paquetes asociados a cada uno del dataset ddos_2")

top_n = df_puertos.sort_values("count", ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.bar(top_n["destination_port"].astype(str), top_n["count"])
plt.yscale("log")  # escala log por la dominancia del puerto 80
plt.xlabel("Puerto de destino")
plt.ylabel("Cantidad de paquetes")
plt.title("Paquetes por puerto de destino - ddos_2")
plt.tight_layout()

# GUARDAR EN SVG
plt.savefig(
    "escala_paquetes_por_puerto.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()



In [ ]:
"""¿Cuántas IPs distintas atacan a la víctima?"""
ips_atacantes = df_ddos_2[df_ddos_2["destination_address"] == "192.168.100.3"]["source_address"].nunique()
print(f"Cantidad de IPs atacantes distintas: {ips_atacantes}")

print(df_ddos_2[df_ddos_2["destination_address"] == "192.168.100.3"]["source_address"].value_counts())

In [ ]:
"""num_incoming.connection_per_destip vs num_incoming.connection_per_sourceip """
df_ataque = df_ddos_2[df_ddos_2["destination_address"] == "192.168.100.3"].copy()
df_ataque["start_time"] = pd.to_datetime(df_ataque["start_time"], unit="s")
df_ataque = df_ataque.sort_values("start_time")

plt.figure(figsize=(12, 6))
plt.plot(df_ataque["start_time"], df_ataque["num_incoming.connection_per_destip"], 
          label="Conexiones entrantes recibidas por el equipo destino", alpha=0.7)
plt.plot(df_ataque["start_time"], df_ataque["num_incoming.connection_per_sourceip"], 
          label="Conexiones entrantes asociadas a cada equipo origen", alpha=0.7)
plt.xlabel("Tiempo")
plt.ylabel("Cantidad de conexiones entrantes")
plt.title("Evolución: conexiones por IP destino vs por IP origen")
plt.legend()
plt.xticks(rotation=0)
plt.tight_layout()

# GUARDAR EN SVG
plt.savefig(
    "conexiones_por_ipdest_vs_iporig.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()

BIBLIOGRAFIA:

What are Protocol Packets and Byte Counts? -. https://www.cbtnuggets.com/blog/technology/networking/understanding-protocol-packets-and-byte-counts

Project 74 : DDoS Attack Classification using Machine Learning-.https://www.youtube.com/watch?v=QqEdSk98Wqs

DDoS Dataset by Devendra -. https://www.kaggle.com/datasets/devendra416/ddos-datasets

DDoS Dataset by SIDDHARTH M -.https://www.kaggle.com/datasets/siddharthm1698/ddos-botnet-attack-on-iot-devices y https://github.com/Chando0185/Multiverse_of_100-_data_science_project_series/blob/main/DDos%20Attack/DDOS%20Attack%20Classification%20Using%20Machine%20Learning.ipynb

Documentacion sobre el ARP (RFC-826) -. https://www.rfc-editor.org/rfc/rfc826.html?utm_source=chatgpt.com